# 🧠 LeetCode 347: Top K Frequent Elements

**Pattern:** Bucket Sort / Heap

- **Created:** 2026-02-28
- **Focus:** Counting frequencies and finding the top performers efficiently
- **Tags:** `array` | `hash-table` | `bucket-sort` | `heap` | `medium` | `citi-prep`

## 📖 Problem Statement

Given an integer array `nums` and an integer `k`, return the `k` most frequent elements. You may return the answer in any order.

**Follow up:** Your algorithm's time complexity must be better than $O(N \log N)$, where $N$ is the array's size.

### Example 1:
- **Input:** `nums = [1,1,1,2,2,3]`, `k = 2`
- **Output:** `[1,2]`
- **Explanation:** `1` occurs 3 times. `2` occurs 2 times. `3` occurs 1 time. The top 2 most frequent are `1` and `2`.

## 🧠 Mental Model & Intuition

**Approach 1: The Sorting Hat (Brute Force)**
Count the frequencies using a tally sheet (Hash Map). Then, sort the tally sheet by the counts from highest to lowest and take the top K. Because you are sorting, this is $O(N \log N)$. The problem explicitly asked us to be faster.

**Approach 2: The Min-Heap (Better)**
Count frequencies. As you go through the counts, push them into a Min-Heap of size `k`. If the heap gets larger than `k`, pop the smallest. After processing all elements, the heap contains the top `k`. $O(N \log K)$ time.

**Approach 3: Bucket Sort (Optimal)**
Create an array of "buckets" where the *index* represents the frequency count, and the *value* corresponds to a list of numbers that have that frequency.
- A number can never appear more than $N$ times exactly.
- So we make an array of size $N+1$.
- If `1` appears 3 times, we put `1` into `bucket[3]`.
- We then scan the buckets backwards (from $N$ down to 1), gathering numbers until we have `k` of them.
- $O(N)$ time. Pure genius.

## 🐢 Approach 1 (Sorting)

Very easy to write in Python, but technically fails the follow-up requirement of beating $O(N \log N)$.

```python
from collections import Counter
def topKFrequentSort(nums, k):
    count = Counter(nums)
    # sorted() sorts keys based on the values (the counts) descending
    return sorted(count.keys(), key=count.get, reverse=True)[:k]
```
**Time:** $O(N \log N)$ | **Space:** $O(N)$

In [ ]:
# Approach 2: Min-Heap
import heapq
from collections import Counter

def topKFrequentHeap(nums: list[int], k: int) -> list[int]:
    # 1. Count frequencies O(N)
    count = Counter(nums)
    
    # 2. Maintain a heap of size K
    # We push tuples of (frequency, num) so heapq sorts by frequency
    heap = []
    for num, freq in count.items():
        heapq.heappush(heap, (freq, num))
        if len(heap) > k:
            heapq.heappop(heap) # Pops the smallest frequency
            
    # 3. Extract results O(K log K)
    return [num for freq, num in heap]

print("Heap approach:", topKFrequentHeap([1,1,1,2,2,3], 2))

In [ ]:
# Approach 3: Optimal Bucket Sort
def topKFrequent(nums: list[int], k: int) -> list[int]:
    count = {} # Hash map for counting
    freq = [[] for _ in range(len(nums) + 1)] # Array of lists (Buckets)
    
    # 1. Count frequencies: O(N)
    for num in nums:
        count[num] = count.get(num, 0) + 1
        
    # 2. Populate the buckets: O(N)
    # The frequency is the INDEX of the array
    for num, c in count.items():
        freq[c].append(num)
        
    # 3. Gather top K elements passing backwards through buckets: O(N)
    res = []
    for i in range(len(freq) - 1, 0, -1): # Start at N, go down to 1
        for num in freq[i]:
            res.append(num)
            if len(res) == k:
                return res
                
print("Bucket Sort approach:", topKFrequent([1,1,1,2,2,3], 2))

## ⏱️ Complexity Analysis (Bucket Sort)

- **Time Complexity:** $O(N)$ because counting takes $N$, placing in buckets takes $N$, and reading back takes at most $N$ operations.
- **Space Complexity:** $O(N)$ because the tally map stores at most $N$ keys, and the `freq` buckets array is exactly size $N+1$.

## 🎤 Interview Q&A

### Q1: Why can we guarantee the Bucket array doesn't need to be larger than N+1?
**Answer:** Because the absolute maximum frequency a number could have is if the *entire array* is just that one number (e.g., `[1, 1, 1, 1]`). In that case, its frequency is exactly $N$. So index $N$ is the highest index we will ever need.

---

### Q2: Between the Min-Heap approach and the Bucket Sort approach, which do you prefer?
**Answer:** In an interview, I would explain the Min-Heap first because it's highly robust and works even when $N$ is extremely large (e.g., streaming data where $N$ is practically infinite and Bucket Sort would blow out memory). However, for a fixed array in memory, Bucket Sort is the optimal $O(N)$ answer the interviewer is specifically fishing for by including the "better than $N \log N$" follow-up.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Bucket Sort** | A sorting algorithm that groups items by a specific attribute (like frequency) into buckets, avoiding comparison loops. | `freq[count].append(num)` |
| **Min-Heap** | A binary tree where the parent node is always smaller than its children. Great for keeping "Top K". | `import heapq` |
| **$O(K \log N)$** | The runtime for inserting $K$ elements into a heap of size $N$. | Common heap benchmark |

## 💼 The Citi Narrative

**Context:** Trade execution error logging.

**Scenario:** In production, we ingested millions of application log events per hour (our array $N$). During a massive system outage, we needed an immediate dashboard showing the Top 10 most frequent Error Codes (our $k$) to diagnose the root cause.

**Impact:** Attempting to `$sort` a map of millions of distinct log codes locked up the CPU. Instead, I implemented a Min-Heap tracker in the aggregation script. By limiting the heap strictly to size $k=10$, we transformed a heavy batch-job into a near real-time $O(N \log K)$ streaming pipeline that successfully emitted the top failures instantly as they occurred.

In [ ]:
# EXERCISE: Trace the bucket array
# Manually define what the freq array looks like for nums=[1,1,2], count={1: 2, 2: 1}
print("freq[0] = [] (Nothing appears 0 times)")
print("freq[1] = [2] (The number '2' appears 1 time)")
print("freq[2] = [1] (The number '1' appears 2 times)")
print("freq[3] = [] (Nothing appears 3 times)")
print("We scan backwards. Hit 2. Result -> [1]. Now we have k=1 items. Return.")

## 🎯 Summary: Key Takeaways

### The Pattern
**Min-Heap / Bucket Sort** — Prioritizing top bounded elements without fully sorting the array.

### When to Use It
- ✅ Leaderboards or 'Trending' items.
- ✅ Extracting highest-value events from logs dynamically.
- ❌ **Don't use when:** Need exact global median or strict sequential order of all N items.

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N \log N)$ | $O(N)$ |
| Optimal | $O(N \log K)$ | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Min-Heap / Bucket Sort** and you've mastered one of the most common patterns in data engineering interviews. 🚀